In [ ]:
import os
import json
import math
import random
import string
import numpy as np
from collections import Counter
from dataclasses import dataclass
from typing import List, Dict, Tuple, Any
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download, login

# Configurations and hyperparameters

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = os.environ.get("HF_TOKEN", None)
if HF_TOKEN:
    login(token=HF_TOKEN)

# Paths for Phase 1 Checkpoint & Tokenizer
TOKENIZER_PATH = "/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json"
CHECKPOINT_PATH = "/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt"

# Phase 2 Hyperparameters
MAX_SEQ_LEN = 512
STRIDE = 128
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 3e-4      # adapter (LoRMA) params
HEAD_LR = 1e-3            # classifier head
EPOCHS = 16
LORA_RANK = 32
LORA_ALPHA = 64.0
USE_VAL = True            # True: train fold1, validate on fold2 (tunes decoding, saves cfg).  False: FINAL run on folds 1+2
MASK_PROB = 0.05          # random <unk> token masking during training (regularizer)
EMA_DECAY = 0.98          # EMA over trainable (adapter/head) weights
INFER_STRIDE = 384        # inference window overlap -> step = 512-384 = 128 (denser averaging)
CFG_PATH = "/kaggle/working/decode_cfg.json"

# 12 Official Labels
LABEL_NAMES = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE",
    "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]
# BIO Scheme (25 tags total)
ID2LABEL = {0: "O"}
LABEL2ID = {"O": 0}
for idx, lbl in enumerate(LABEL_NAMES):
    ID2LABEL[1 + 2 * idx] = f"B-{lbl}"
    LABEL2ID[f"B-{lbl}"] = 1 + 2 * idx
    ID2LABEL[2 + 2 * idx] = f"I-{lbl}"
    LABEL2ID[f"I-{lbl}"] = 2 + 2 * idx
NUM_CLASSES = len(ID2LABEL)
IGNORE_INDEX = -100
print(f"Device: {DEVICE} | Classes: {NUM_CLASSES} | USE_VAL={USE_VAL}")

In [ ]:
SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
END_OF_WORD = "</w>"

class ByteBPETokenizer:
    def __init__(self, vocab_size=8000):
        self.vocab_size = vocab_size
        self.merges = []
        self.merge_rank = {}
        self.itos = []
        self.stoi = {}

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(vocab_size=data["vocab_size"])
        tok.itos = data["itos"]
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok.merges = [tuple(p) for p in data["merges"]]
        tok.merge_rank = {p: i for i, p in enumerate(tok.merges)}
        return tok

    def _word_to_symbols(self, word):
        return list(word.encode("utf-8").decode("latin-1")) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        symbols = self._word_to_symbols(word)
        while len(symbols) > 1:
            pairs_here = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            candidates = [(self.merge_rank[p], idx) for idx, p in enumerate(pairs_here) if p in self.merge_rank]
            if not candidates:
                break
            _, idx = min(candidates)
            merged = symbols[idx] + symbols[idx + 1]
            symbols = symbols[:idx] + [merged] + symbols[idx + 2:]
        return symbols

    def encode_with_exact_offsets(self, text: str) -> Tuple[List[int], List[Tuple[int, int]]]:
        """
        Calculates EXACT character-level [start, end) offsets for every single subword token.
        """
        tokens = []
        offsets = []
        i = 0
        n = len(text)

        while i < n:
            while i < n and text[i].isspace():
                i += 1
            if i >= n:
                break
            word_start = i
            while i < n and not text[i].isspace():
                i += 1
            word_end = i
            word = text[word_start:word_end]

            subwords = self._apply_merges_to_word(word)
            curr_pos = word_start

            for sym in subwords:
                clean_sym = sym.replace(END_OF_WORD, "")
                try:
                    sym_chars = clean_sym.encode("latin-1").decode("utf-8", errors="replace")
                except Exception:
                    sym_chars = clean_sym

                sub_len = len(sym_chars)
                sub_start = curr_pos
                sub_end = min(word_end, curr_pos + sub_len)
                curr_pos = sub_end

                t_id = self.stoi.get(sym, self.stoi.get("<unk>", 1))
                tokens.append(t_id)
                offsets.append((sub_start, sub_end))

        return tokens, offsets

# Load tokenizer
tokenizer = ByteBPETokenizer.load(TOKENIZER_PATH)
print(f"Loaded tokenizer | Vocab size: {tokenizer.vocab_size}")

In [ ]:
import zipfile
import os

source_dir = '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt'
target_pt = '/kaggle/working/decoder_lm.pt'

# Re-package the folder into the single PyTorch archive file
with zipfile.ZipFile(target_pt, 'w', compression=zipfile.ZIP_STORED) as zf:
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            full_path = os.path.join(root, file)
            # relative path inside the archive
            rel_path = os.path.relpath(full_path, source_dir)
            zf.write(full_path, arcname=rel_path)

print(f"✓ Created valid checkpoint file: {target_pt} ({os.path.getsize(target_pt)/1e6:.1f} MB)")

# Set CHECKPOINT_PATH to this working file
CHECKPOINT_PATH = target_pt

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def get_alibi_slopes(n_heads: int):
    def get_slopes_power_of_2(n):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        ratio = start
        return [start * (ratio ** i) for i in range(n)]

    if math.log2(n_heads).is_integer():
        return torch.tensor(get_slopes_power_of_2(n_heads))
    else:
        closest_pow2 = 2 ** math.floor(math.log2(n_heads))
        slopes_a = get_slopes_power_of_2(closest_pow2)
        slopes_b = get_slopes_power_of_2(2 * closest_pow2)[0::2][:n_heads - closest_pow2]
        return torch.tensor(slopes_a + slopes_b)

class LORMALinear(nn.Module):
    def __init__(self, original_linear: nn.Linear, rank: int = 16, alpha: float = 32.0):
        super().__init__()
        self.original_linear = original_linear
        self.rank = rank
        self.scaling = alpha / rank
        self.original_linear.weight.requires_grad = False

        in_f = original_linear.in_features
        out_f = original_linear.out_features
        self.lorma_down = nn.Linear(in_f, rank, bias=False)
        self.lorma_up = nn.Linear(rank, out_f, bias=False)

        nn.init.kaiming_uniform_(self.lorma_down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_up.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.original_linear(x) + self.lorma_up(self.lorma_down(x)) * self.scaling

class ALiBiBidirectionalAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)

        slopes = get_alibi_slopes(n_heads).unsqueeze(1).unsqueeze(2)
        self.register_buffer("slopes", slopes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, _ = x.shape
        q = self.q_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)

        # Symmetric bidirectional distance
        pos = torch.arange(S, device=x.device)
        dist = -torch.abs(pos.unsqueeze(0) - pos.unsqueeze(1)).float()
        alibi_bias = self.slopes * dist.unsqueeze(0)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=alibi_bias,
            dropout_p=self.dropout if self.training else 0.0
        )
        out = out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        return self.out_proj(out)

class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = ALiBiBidirectionalAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class Arch4LNERModel(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 640, n_layers: int = 16,
                 n_heads: int = 8, d_ff: int = 2560, num_classes: int = NUM_CLASSES,
                 dropout: float = 0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.drop(self.tok_emb(input_ids))
        for block in self.blocks:
            x = block(x)
        x = self.norm_f(x)
        return self.classifier(x)

    def apply_lorma(self, rank: int = 16, alpha: float = 32.0, device=DEVICE):
        for param in self.parameters():
            param.requires_grad = False

        for block in self.blocks:
            block.attn.q_proj = LORMALinear(block.attn.q_proj, rank=rank, alpha=alpha).to(device)
            block.attn.v_proj = LORMALinear(block.attn.v_proj, rank=rank, alpha=alpha).to(device)
            block.ffn.gate_proj = LORMALinear(block.ffn.gate_proj, rank=rank, alpha=alpha).to(device)
            block.ffn.up_proj = LORMALinear(block.ffn.up_proj, rank=rank, alpha=alpha).to(device)

        for param in self.classifier.parameters():
            param.requires_grad = True

# ---- NEW: adapters on every linear layer ----
def apply_lorma_all(model, rank, alpha, device):
    for p in model.parameters():
        p.requires_grad = False
    for b in model.blocks:
        for n in ["q_proj", "k_proj", "v_proj", "out_proj"]:
            setattr(b.attn, n, LORMALinear(getattr(b.attn, n), rank, alpha).to(device))
        for n in ["gate_proj", "up_proj", "down_proj"]:
            setattr(b.ffn, n, LORMALinear(getattr(b.ffn, n), rank, alpha).to(device))
    for p in model.classifier.parameters():
        p.requires_grad = True
    for p in model.norm_f.parameters():
        p.requires_grad = True

# Instantiate & Load Checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
d_model_ckpt = state_dict["tok_emb.weight"].shape[1]
vocab_size_ckpt = state_dict["tok_emb.weight"].shape[0]
block_indices = [int(k.split(".")[1]) for k in state_dict.keys() if k.startswith("blocks.") and k.split(".")[1].isdigit()]
n_layers_ckpt = max(block_indices) + 1 if block_indices else 16
n_heads = 8 if d_model_ckpt == 640 else 12

model = Arch4LNERModel(vocab_size=vocab_size_ckpt, d_model=d_model_ckpt, n_layers=n_layers_ckpt, n_heads=n_heads).to(DEVICE)
backbone_dict = {k: v for k, v in state_dict.items() if not k.startswith("lm_head") and not k.startswith("head")}
missing, unexpected = model.load_state_dict(backbone_dict, strict=False)
print("MISSING   :", len(missing), missing[:8])
print("UNEXPECTED:", len(unexpected), unexpected[:8])
assert all(m.startswith("classifier.") or m.endswith(".slopes") for m in missing), \
    "Backbone weights did NOT load - fix key names before continuing!"

apply_lorma_all(model, LORA_RANK, LORA_ALPHA, DEVICE)
model.to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Params: {total_params:,} | Trainable Params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

In [ ]:
def download_phase2_file(filename: str) -> str:
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

print("Downloading dataset files...")
f1_text_path = download_phase2_file("Phase_2_LNER/Train/fold1-text.json")
f1_gold_path = download_phase2_file("Phase_2_LNER/Train/fold1-gold.json")
f2_text_path = download_phase2_file("Phase_2_LNER/Train/fold2-text.json")
f2_gold_path = download_phase2_file("Phase_2_LNER/Train/fold2-gold.json")
f3_text_path = download_phase2_file("Phase_2_LNER/Test/fold3-text.json")

with open(f1_text_path, "r", encoding="utf-8") as f: train_texts = json.load(f)
with open(f1_gold_path, "r", encoding="utf-8") as f: train_golds = json.load(f)
with open(f2_text_path, "r", encoding="utf-8") as f: val_texts = json.load(f)
with open(f2_gold_path, "r", encoding="utf-8") as f: val_golds = json.load(f)
with open(f3_text_path, "r", encoding="utf-8") as f: test_texts = json.load(f)

if USE_VAL:
    tr_texts, tr_golds = train_texts, train_golds
else:
    tr_texts = {**train_texts, **val_texts}
    tr_golds = {**train_golds, **val_golds}
print(f"Training docs: {len(tr_texts)} | USE_VAL={USE_VAL}")

# windows; the last one is always full-length. `offset` lets us jitter window boundaries each epoch.
def window_starts(n, max_len=MAX_SEQ_LEN, stride=STRIDE, offset=0):
    if n <= max_len:
        return [0]
    step = max_len - stride
    starts = {0, n - max_len}
    starts.update(range(offset, n - max_len + 1, step))
    return sorted(starts)

def tokens_and_labels(text, annots, tok):
    token_ids, offsets = tok.encode_with_exact_offsets(text)
    token_labels = [0] * len(token_ids)
    for annot in annots:
        g_start, g_end, g_lbl = annot[0], annot[1], annot[2]
        first_matched = False
        for idx, (t_s, t_e) in enumerate(offsets):
            if max(t_s, g_start) < min(t_e, g_end):
                if not first_matched:
                    token_labels[idx] = LABEL2ID.get(f"B-{g_lbl}", 0)
                    first_matched = True
                else:
                    token_labels[idx] = LABEL2ID.get(f"I-{g_lbl}", 0)
    # zero-length tokens (e.g. a stand-alone "</w>") carry no characters: never train on them as "O"
    for idx, (t_s, t_e) in enumerate(offsets):
        if t_e <= t_s:
            token_labels[idx] = IGNORE_INDEX
    return token_ids, offsets, token_labels

class ExactLNERDataset(Dataset):
    def __init__(self, docs, golds, tokenizer, max_len=MAX_SEQ_LEN, stride=STRIDE):
        self.max_len, self.stride = max_len, stride
        self.docs = []
        for doc_id, text in docs.items():
            ids, _, lbls = tokens_and_labels(text, golds.get(doc_id, []), tokenizer)
            if ids:
                self.docs.append((ids, lbls))
        self.samples = []
        self.resample(jitter=False)

    def resample(self, jitter=True):
        """Rebuild windows. With jitter, window boundaries shift randomly every epoch (cheap augmentation)."""
        step = self.max_len - self.stride
        self.samples = []
        for ids, lbls in self.docs:
            off = random.randrange(step) if jitter else 0
            for s in window_starts(len(ids), self.max_len, self.stride, off):
                e = min(len(ids), s + self.max_len)
                pad = self.max_len - (e - s)
                self.samples.append({
                    "input_ids": torch.tensor(ids[s:e] + [0] * pad, dtype=torch.long),
                    "labels": torch.tensor(lbls[s:e] + [IGNORE_INDEX] * pad, dtype=torch.long),
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

train_ds = ExactLNERDataset(tr_texts, tr_golds, tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
print(f"Total training sliding windows: {len(train_ds)}")

In [ ]:
import re

# ---------------- boundary cleanup (mode is tuned on validation) ----------------
WS = " \t\n\r"
QUOTES = "\u2018\u2019\u201c\u201d\"'"
BULLETS = "\u2013\u2014\u2022\u00b7\u2026"
FULL_CHARS = WS + string.punctuation + QUOTES + BULLETS
LIGHT_CHARS = WS + ",;:-" + QUOTES + BULLETS
CLEAN_MODES = ["full", "smart", "light", "ws", "none"]

def clean_span(text, start, end, label, mode="full"):
    if mode == "none":
        return start, end, label
    chars = WS if mode == "ws" else (FULL_CHARS if mode == "full" else LIGHT_CHARS)
    changed = True
    while changed:
        changed = False
        while start < end and text[start] in chars:
            start += 1; changed = True
        while end > start and text[end - 1] in chars:
            end -= 1; changed = True
        if mode == "smart" and end > start:
            seg = text[start:end]
            # trailing '.' only removed after a long non-abbreviation word ("Singh." -> "Singh", keep "Ltd." / "U.P.")
            if seg.endswith("."):
                last = seg[:-1].split()[-1] if seg[:-1].split() else ""
                if len(last) >= 4 and "." not in last:
                    end -= 1; changed = True; continue
            # unbalanced brackets at the edges
            if seg[-1] in ")]" and seg.count("(") + seg.count("[") < seg.count(")") + seg.count("]"):
                end -= 1; changed = True; continue
            if seg[0] in "([" and seg.count("(") + seg.count("[") > seg.count(")") + seg.count("]"):
                start += 1; changed = True; continue
    return start, end, label

def decode_spans_exact(doc_text, pred_ids, offsets, clean="full"):
    spans, cur_l, cs, ce = [], None, -1, -1

    def flush():
        if cur_l is not None:
            s, e, l = clean_span(doc_text, cs, ce, cur_l, clean)
            if e > s:
                spans.append([s, e, l])

    for t_id, (s_c, e_c) in zip(pred_ids, offsets):
        if e_c <= s_c:            # zero-length token: ignore so it cannot split a span
            continue
        tag = ID2LABEL.get(int(t_id), "O")
        if tag.startswith("B-"):
            flush(); cur_l, cs, ce = tag[2:], s_c, e_c
        elif tag.startswith("I-") and cur_l == tag[2:]:
            ce = e_c
        elif tag.startswith("I-"):
            flush(); cur_l, cs, ce = tag[2:], s_c, e_c
        else:
            flush(); cur_l = None
    flush()

    seen, out = set(), []
    for s in spans:
        if tuple(s) not in seen:
            seen.add(tuple(s)); out.append(s)
    return out

# ---------------- BIO-constrained Viterbi (with continuation bonus) ----------------
NEG = -1e9
ALLOWED = np.ones((NUM_CLASSES, NUM_CLASSES), dtype=bool)     # [prev, cur]
for cur in range(NUM_CLASSES):
    if ID2LABEL[cur].startswith("I-"):
        for prev in range(NUM_CLASSES):
            ALLOWED[prev, cur] = (ID2LABEL[prev] != "O") and (ID2LABEL[prev][2:] == ID2LABEL[cur][2:])
START_OK = np.array([not ID2LABEL[c].startswith("I-") for c in range(NUM_CLASSES)])

def make_trans(cont_bonus=0.0):
    t = np.where(ALLOWED, 0.0, NEG).astype(np.float64)
    for cur in range(NUM_CLASSES):
        if ID2LABEL[cur].startswith("I-"):
            for prev in range(NUM_CLASSES):
                if ALLOWED[prev, cur]:
                    t[prev, cur] += cont_bonus
    return t

def viterbi(logp, trans):
    T, C = logp.shape
    score = logp[0].copy(); score[~START_OK] = NEG
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + trans
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]

def merge_spans(text, spans, max_gap=0):
    out = []
    for s, e, l in sorted(spans):
        if out and out[-1][2] == l and 0 <= s - out[-1][1] <= max_gap and text[out[-1][1]:s].strip() == "":
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e, l])
    return out

# ---------------- document-level consistency post-processing ----------------
def postprocess(text, spans, unify=False, prop_labels=(), min_len=4):
    """unify: identical surface strings get the majority label.
       prop_labels: labels whose predicted strings are also tagged at every other (unclaimed) occurrence in the doc."""
    if not spans or (not unify and not prop_labels):
        return spans
    spans = [list(s) for s in spans]
    if unify:
        votes = {}
        for s, e, l in spans:
            votes.setdefault(text[s:e], Counter())[l] += 1
        for sp in spans:
            c = votes[text[sp[0]:sp[1]]]
            top = max(c.values())
            if c[sp[2]] != top:
                sp[2] = next(l for l, v in c.most_common() if v == top)
    if prop_labels:
        occupied = [(s, e) for s, e, _ in spans]
        cands = {}
        for s, e, l in spans:
            if l in prop_labels and e - s >= min_len:
                cands.setdefault(text[s:e], Counter())[l] += 1
        new = []
        for t in sorted(cands, key=len, reverse=True):
            lbl = cands[t].most_common(1)[0][0]
            for m in re.finditer(r"(?<!\w)" + re.escape(t) + r"(?!\w)", text):
                ms, me = m.span()
                if any(ms < oe and os_ < me for os_, oe in occupied):
                    continue
                new.append([ms, me, lbl]); occupied.append((ms, me))
        spans += new
    return sorted(spans)

# ---------------- per-document log-probs: edge-aware weighted average over overlapping windows ----------------
@torch.no_grad()
def doc_logp(text):
    model.eval()
    ids, offs = tokenizer.encode_with_exact_offsets(text)
    if not ids:
        return None, None
    n = len(ids)
    acc = torch.zeros(n, NUM_CLASSES); wsum = torch.zeros(n, 1)
    for s in window_starts(n, MAX_SEQ_LEN, INFER_STRIDE):
        e = min(n, s + MAX_SEQ_LEN); L = e - s
        chunk = ids[s:e]
        x = torch.tensor([chunk + [0] * (MAX_SEQ_LEN - L)], dtype=torch.long, device=DEVICE)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            lp = F.log_softmax(model(x)[0].float(), dim=-1)[:L].cpu()
        pos = torch.arange(L, dtype=torch.float32)
        left = pos + 1 if s > 0 else torch.full((L,), 1e9)      # tokens near an *interior* window edge get less weight
        right = (L - pos) if e < n else torch.full((L,), 1e9)   # true document edges are not penalised
        w = torch.minimum(left, right).clamp(max=64.0) / 64.0
        acc[s:e] += lp * w[:, None]; wsum[s:e] += w[:, None]
    return (acc / wsum).numpy(), offs

# ---------------- decoding config ----------------
DEFAULT_CFG = dict(o_bias=2.0, cont=2.0, clean="full", max_gap=0, label_bias={}, unify=False, prop=[])

def decode_cached(cache, texts, cfg, post=True):
    trans, out = make_trans(cfg["cont"]), {}
    for d, (lp, offs) in cache.items():
        if lp is None:
            out[d] = []
            continue
        keep = np.array([e > s for s, e in offs])           # drop zero-length tokens before Viterbi
        offs_k = [o for o, k in zip(offs, keep) if k]
        lp = lp[keep].astype(np.float64).copy()
        lp[:, 0] += cfg["o_bias"]
        for lbl, b in cfg.get("label_bias", {}).items():
            k = LABEL_NAMES.index(lbl)
            lp[:, 1 + 2 * k] += b; lp[:, 2 + 2 * k] += b
        sp = decode_spans_exact(texts[d], viterbi(lp, trans), offs_k, cfg["clean"])
        if cfg["max_gap"] > 0:
            sp = merge_spans(texts[d], sp, cfg["max_gap"])
        if post:
            sp = postprocess(texts[d], sp, cfg.get("unify", False), tuple(cfg.get("prop", [])))
        out[d] = sp
    return out

def strict_macro_f1(pred, gold, labels=LABEL_NAMES):
    f1s = []
    for lbl in labels:
        tp = fp = fn = 0
        for d in gold:
            g = {(a[0], a[1]) for a in gold[d] if a[2] == lbl}
            p = {(a[0], a[1]) for a in pred.get(d, []) if a[2] == lbl}
            tp += len(g & p); fp += len(p - g); fn += len(g - p)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return sum(f1s) / len(f1s), dict(zip(labels, f1s))

In [ ]:
# ---- class weights: O = 1, rare tags up to 20x ----
counts = torch.zeros(NUM_CLASSES)
for s in train_ds.samples:
    l = s["labels"]; l = l[l != IGNORE_INDEX]
    counts += torch.bincount(l, minlength=NUM_CLASSES).float()
w = (counts.sum() / counts.clamp(min=1)) ** 0.5
w = (w / w[0]).clamp(max=20.0)
print("O fraction:", round((counts[0] / counts.sum()).item(), 4))
criterion = nn.CrossEntropyLoss(weight=w.to(DEVICE), ignore_index=IGNORE_INDEX)

# ---- optimizer with a separate (higher) LR for the head ----
head_params = list(model.classifier.parameters())
head_ids = {id(p) for p in head_params}
other_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
optimizer = torch.optim.AdamW([
    {"params": other_params, "lr": LEARNING_RATE},
    {"params": head_params,  "lr": HEAD_LR},
], weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
warmup = max(1, int(0.05 * total_steps))
def lr_lambda(s):
    if s < warmup:
        return (s + 1) / warmup
    prog = min(1.0, (s - warmup) / max(1, total_steps - warmup))
    return 0.03 + 0.97 * 0.5 * (1 + math.cos(math.pi * prog))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ---- EMA of trainable weights (evaluation / final model use the EMA copy) ----
trainable_named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
ema = {n: p.detach().clone() for n, p in trainable_named}
ema_steps = 0
def ema_update():
    global ema_steps
    ema_steps += 1
    d = min(EMA_DECAY, (1 + ema_steps) / (10 + ema_steps))
    with torch.no_grad():
        for n, p in trainable_named:
            ema[n].mul_(d).add_(p.detach(), alpha=1 - d)

@torch.no_grad()
def load_named(state):
    for n, p in model.named_parameters():
        if n in state:
            p.data.copy_(state[n])

UNK_ID = tokenizer.stoi.get("<unk>", 1)

def eval_val_ema():
    backup = {n: p.detach().clone() for n, p in trainable_named}
    load_named(ema)
    vc = {d: doc_logp(t) for d, t in val_texts.items()}
    f1, per = strict_macro_f1(decode_cached(vc, val_texts, DEFAULT_CFG), val_golds)
    load_named(backup)
    return f1, per

print(f"Training {EPOCHS} epochs | USE_VAL={USE_VAL}")
best_f1, best_state, best_epoch = -1.0, None, 0

for epoch in range(EPOCHS):
    model.train()
    train_ds.resample(jitter=True)          # new window boundaries every epoch
    running, n_log = 0.0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        if MASK_PROB > 0:                    # random token masking -> forces use of context, not just surface forms
            m = (torch.rand(input_ids.shape, device=DEVICE) < MASK_PROB) & (input_ids != 0)
            input_ids = input_ids.masked_fill(m, UNK_ID)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(input_ids)
        loss = criterion(logits.float().view(-1, NUM_CLASSES), labels.view(-1))

        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        running += loss.item(); n_log += 1

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            ema_update()

        if (step + 1) % 100 == 0:
            print(f"Epoch [{epoch + 1}/{EPOCHS}] | Step [{step + 1}/{len(train_loader)}] | Loss: {running / n_log:.4f}")
            running, n_log = 0.0, 0

    if USE_VAL and ((epoch + 1) % 2 == 0 and epoch + 1 >= 4):
        f1, per = eval_val_ema()
        print(f"==> Epoch {epoch + 1} | VAL strict macro-F1 (EMA, default decode): {f1:.4f}")
        print({k: round(v, 3) for k, v in per.items()})
        if f1 > best_f1:
            best_f1, best_epoch = f1, epoch + 1
            best_state = {k: v.clone() for k, v in ema.items()}

# final weights = EMA copy (best-on-val EMA when validating, last EMA for the final folds1+2 run)
if USE_VAL and best_state is not None:
    load_named(best_state)
    print(f"Restored best EMA from epoch {best_epoch} (val F1 {best_f1:.4f})")
else:
    load_named(ema)
    print("Loaded final EMA weights")
print("✓ Training finished!")

In [ ]:
if USE_VAL:
    val_cache = {d: doc_logp(t) for d, t in val_texts.items()}
    def score(cfg, post=True):
        return strict_macro_f1(decode_cached(val_cache, val_texts, cfg, post), val_golds)

    # --- diagnostic: ceiling imposed by tokenisation + span cleanup (gold labels -> tokens -> spans) ---
    print("Oracle F1 (gold tags decoded with each cleanup mode; 1.0 = no boundary loss):")
    for mode in CLEAN_MODES:
        orc = {}
        for d, t in val_texts.items():
            _, offs, lbls = tokens_and_labels(t, val_golds.get(d, []), tokenizer)
            orc[d] = decode_spans_exact(t, lbls, offs, mode)
        print(f"  {mode:6s} -> {strict_macro_f1(orc, val_golds)[0]:.4f}")

    CFG = {**DEFAULT_CFG, "label_bias": {}, "prop": []}

    # stage 1: O bias x continuation bonus
    best = (-1.0, None)
    for ob in [1.0, 2.0, 3.0, 4.0]:
        for cb in [1.0, 2.0, 3.0, 4.0, 6.0]:
            f, _ = score({**CFG, "o_bias": ob, "cont": cb}, post=False)
            print(f"o_bias={ob} cont={cb} -> {f:.4f}")
            if f > best[0]: best = (f, (ob, cb))
    CFG["o_bias"], CFG["cont"] = best[1]
    print("stage1:", best)

    # stage 2: boundary cleanup mode
    best = (-1.0, None)
    for mode in CLEAN_MODES:
        f, _ = score({**CFG, "clean": mode}, post=False)
        print(f"clean={mode} -> {f:.4f}")
        if f > best[0]: best = (f, mode)
    CFG["clean"] = best[1]
    print("stage2:", best)

    # stage 3: per-label bias (only kept if it clearly helps that label)
    _, base_per = score(CFG, post=False)
    for lbl in LABEL_NAMES:
        best_b, best_lf = 0.0, base_per[lbl]
        for b in [-1.0, -0.5, 0.5, 1.0, 1.5]:
            _, per = score({**CFG, "label_bias": {**CFG["label_bias"], lbl: b}}, post=False)
            if per[lbl] > best_lf + 0.015:
                best_b, best_lf = b, per[lbl]
        if best_b != 0.0:
            CFG["label_bias"][lbl] = best_b
            print(f"  label_bias[{lbl}] = {best_b}  ({base_per[lbl]:.3f} -> {best_lf:.3f})")
    print("stage3 F1:", round(score(CFG, post=False)[0], 4))

    # stage 4: merge gap
    best = (score(CFG, post=False)[0], 0)
    for g in [1, 2]:
        f, _ = score({**CFG, "max_gap": g}, post=False)
        print(f"max_gap={g} -> {f:.4f}")
        if f > best[0] + 0.001: best = (f, g)
    CFG["max_gap"] = best[1]

    # stage 5: document-level consistency (majority label per string; propagate to missed repeats)
    base_spans = decode_cached(val_cache, val_texts, CFG, post=False)
    def post_score(unify, prop):
        pr = {d: postprocess(val_texts[d], base_spans[d], unify, tuple(prop)) for d in base_spans}
        return strict_macro_f1(pr, val_golds)
    f0, per0 = post_score(False, [])
    f_u, _ = post_score(True, [])
    CFG["unify"] = bool(f_u > f0 + 0.002)
    print(f"unify: {f0:.4f} -> {f_u:.4f}  (use={CFG['unify']})")
    f0, per0 = post_score(CFG["unify"], [])
    for lbl in LABEL_NAMES:
        _, per = post_score(CFG["unify"], [lbl])
        if per[lbl] > per0[lbl] + 0.015:
            CFG["prop"].append(lbl)
            print(f"  propagate[{lbl}]: {per0[lbl]:.3f} -> {per[lbl]:.3f}")

    final_f, final_per = score(CFG, post=True)
    print(f"\nFINAL CFG: {CFG}\nval strict macro-F1 = {final_f:.4f}")
    print({k: round(v, 3) for k, v in final_per.items()})
    with open(CFG_PATH, "w") as f:
        json.dump(CFG, f)
    print(f"saved -> {CFG_PATH}   (copy this CFG into DEFAULT_CFG-style dict if you start a fresh session for the final run)")
else:
    # FINAL run on folds 1+2: reuse the decoding config tuned during the USE_VAL=True run
    if os.path.exists(CFG_PATH):
        with open(CFG_PATH) as f: CFG = json.load(f)
        print("Loaded tuned decode cfg:", CFG)
    else:
        CFG = dict(DEFAULT_CFG)
        print("WARNING: no saved cfg found; paste the tuned CFG from the USE_VAL=True run here. Using DEFAULT_CFG:", CFG)

In [ ]:
test_cache = {d: doc_logp(t) for d, t in test_texts.items()}
test_predictions = decode_cached(test_cache, test_texts, CFG)
test_predictions = {d: [[int(a[0]), int(a[1]), str(a[2])] for a in sp] for d, sp in test_predictions.items()}

output_json = "/kaggle/working/prediction.json"
output_zip = "/kaggle/working/submission.zip"
with open(output_json, "w", encoding="utf-8") as f:
    f.write(json.dumps(test_predictions))
    f.flush()
    os.fsync(f.fileno())
if os.path.exists(output_zip):
    os.remove(output_zip)
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(output_json, arcname="prediction.json")
print(f"\n✓ Submission zip created: {output_zip} ({os.path.getsize(output_zip):,} bytes)")

with zipfile.ZipFile(output_zip, "r") as zf:
    verified = json.loads(zf.read("prediction.json").decode("utf-8"))
    first_k = list(verified.keys())[0]
    print(f"✓ ZIP VERIFIED: Found {len(verified)} documents.")
    print(f"Sample prediction for doc {first_k}: {verified[first_k][:3]}")